# E24 — quantas barras eu ergui antes de acreditar numa

O capítulo 8 compara **quatro sinais dentro de dez faixas** e julga cada comparação contra uma
barra de dois desvios. A barra está certa para uma comparação; a pergunta deste caderno é o que
ela vale para a **bateria**.

A bateria roda aqui dentro dos mundos que **nunca mudam** — o nulo do próprio capítulo —, onde
toda travessia é por acaso.

In [1]:
# <- brinque com: MUNDOS, SEMENTE, DIAS, INFLACAO, FAIXAS
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import graficos, multiplicidade as mult

RAIZ = Path.cwd()
MUNDOS, SEMENTE, DIAS = 60, mult.SEMENTE_PADRAO, mult.DIAS_PADRAO
INFLACAO, FAIXAS = mult.INFLACAO_PADRAO, mult.FAIXAS_PADRAO
M = mult.comparacoes(faixas=FAIXAS)

print("frevolab %s | %d comparacoes (%d sinais x %d faixas) | barra de %.1f desvios"
      % (frevolab.VERSAO, M, mult.SINAIS_PADRAO, FAIXAS, INFLACAO))
print("a conta exata da bateria: %.4f de chance de ao menos uma travessia" % mult.fwer_exata(M))
print("e a barra que mantem os mesmos 5%% no conjunto: %.2f desvios" % mult.quantil_da_familia(M))

frevolab 0.1.0 | 40 comparacoes (4 sinais x 10 faixas) | barra de 2.0 desvios
a conta exata da bateria: 0.8715 de chance de ao menos uma travessia
e a barra que mantem os mesmos 5% no conjunto: 3.22 desvios


In [2]:
# A bateria dentro dos mundos que nunca mudam.
r = mult.bateria(mundos=MUNDOS, semente=SEMENTE, dias=DIAS, faixas=FAIXAS, inflacao=INFLACAO)
print("mundos: %d | comparacoes por mundo: %d" % (r["mundos"], r["comparacoes"]))
print("travessias por mundo: media %.3f | maximo %d" % (r["travessias_media"], r["travessias_maxima"]))
print("taxa por comparacao: %.4f" % r["taxa_por_comparacao"])
print("taxa da bateria (ao menos uma travessia no mundo): %.4f" % r["taxa_da_bateria"])
print("mundos com alguma travessia: %d de %d" % (r["mundos_com_alguma"], r["mundos"]))

mundos: 60 | comparacoes por mundo: 40
travessias por mundo: media 2.267 | maximo 6
taxa por comparacao: 0.0567
taxa da bateria (ao menos uma travessia no mundo): 0.9500
mundos com alguma travessia: 57 de 60


In [3]:
# Figura 1: a barra da comparacao e a barra da bateria.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
ms = np.arange(1, 81)
eixo.plot(ms, [100 * mult.fwer_exata(m) for m in ms], lw=1.8, color="#1f4e79",
          label="a chance de ao menos uma travessia, por acaso")
eixo.axhline(5, color="#333333", ls=":", lw=1.2, label="os 5% de uma comparação só")
eixo.axvline(M, color="#b03a2e", ls="--", lw=1.4, label="a bateria do capítulo: %d comparações" % M)
eixo.scatter([M], [100 * mult.fwer_exata(M)], s=90, marker="o", color="#b03a2e", zorder=5,
             label="%.0f%% em %d comparações" % (100 * mult.fwer_exata(M), M))
eixo.scatter([M], [100 * r["taxa_da_bateria"]], s=90, marker="s", color="#2e7d32", zorder=5,
             label="medido nos mundos que não mudam: %.0f%%" % (100 * r["taxa_da_bateria"]))
eixo.set_xlabel("número de comparações na bateria")
eixo.set_ylabel("chance de ao menos uma travessia (%)")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E24_multiplicidade", 1)
plt.close(fig)
print("figura gravada")

figura gravada


## Leitura visual das figuras

Feita nesta sessão abrindo o .png com a ponte de visão (AGENTS.md §9), depois de o caderno rodar.

O que o desenho mostra, e o que só se vê olhando: o eixo horizontal vai de 0 a 80 comparações e o
vertical, de 0 a 100%; a curva sobe depressa nas primeiras comparações --- em quatro comparações já
passou dos 5% da linha pontilhada --- e achata depois, chegando perto de 100% no fim do eixo. Na
linha tracejada vertical, que são as 40 comparações do capítulo, ficam os dois marcadores: o
círculo vermelho em 87% (a conta exata, com as comparações independentes) e o quadrado verde em
95% (o medido nos mundos que não mudam). **Os dois não coincidem**, e é isso que o desenho ensina:
o verde está oito pontos acima do vermelho, porque as comparações da bateria não são independentes
--- os quatro sinais leem os mesmos dias, e as dez faixas são pedaços ordenados do mesmo perfil.
A dependência aumenta a taxa da bateria, e a conta independente é o piso, não o valor.

In [4]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "multiplicidade_comparacoes": int(M),
    "multiplicidade_sinais": int(mult.SINAIS_PADRAO),
    "multiplicidade_faixas": int(FAIXAS),
    "multiplicidade_barra": float(INFLACAO),
    "multiplicidade_por_comparacao_pct": 100.0 * float(5.0 / 100.0),
    "multiplicidade_fwer_exata_pct": 100.0 * mult.fwer_exata(M),
    "multiplicidade_quantil_familia": float(mult.quantil_da_familia(M)),
    "multiplicidade_mundos": int(r["mundos"]),
    "multiplicidade_dias": int(DIAS),
    "multiplicidade_travessias_media": float(r["travessias_media"]),
    "multiplicidade_travessias_maxima": int(r["travessias_maxima"]),
    "multiplicidade_taxa_comparacao_pct": 100.0 * float(r["taxa_por_comparacao"]),
    "multiplicidade_taxa_bateria_pct": 100.0 * float(r["taxa_da_bateria"]),
    "multiplicidade_mundos_com_alguma": int(r["mundos_com_alguma"]),
}
caminho = Path("lab/resultados/E24_multiplicidade.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E24_multiplicidade.json gravado | 14 grandezas
